# Notebook 3 — Privacy with Anonymizer

**Hands-on synthetic data pipeline workshop with NeMo Data Designer and NeMo Anonymizer — PyData London 2026**

## The setup

Imagine the rich-business-document VLM whose training data we built in Notebook 2 is now deployed. Real users are uploading their internal documents — clinical trial status reports, financial variance memos, product launch readiness plans, operations dashboard exports — and asking questions, and the team wants to study **how the model is being used**: what tasks people actually attempt, where the model struggles, what kinds of documents arrive in the wild.

There's just one problem: in any regulated environment — healthcare, finance, government, or even a large company's internal AI deployment — **most of the team that wants to study these logs cannot directly read them**. The user's session names them, their company, their colleagues, and the project they're working on; the document they uploaded names organisations, principal investigators, owners, dollar variances, KPI targets; the model's reply quotes those identifiers back and pulls in additional ones via tool calls (CRM lookups, owner-contact lookups, compliance checks). Even a "just-debugging" dive turns into a privacy review.

This is exactly the gap [NeMo Anonymizer](https://nvidia-nemo.github.io/Anonymizer/latest/) closes. Detect identifiers, replace them according to a strategy you choose, and produce an audit-safe version of the same dataset that a much wider audience can work with — without watering down the underlying patterns you care about studying.

## What you'll learn

1. **PII detection as a pipeline stage** — GLiNER NER plus an LLM augmenter/validator, both running on hosted endpoints.
2. **Four Replace-mode strategies** — `Substitute`, `Redact`, `Annotate`, `Hash` — and the privacy/utility/audience tradeoffs between them.
3. **`preview()` before committing** — the Rich highlight view shows you exactly what the detector caught (and what it didn't) before you spend tokens on the full dataset.
4. **`data_summary` matters** — the cheapest quality lever Anonymizer gives you. Always set it.
5. **Replace vs. Rewrite** — when per-entity replacement is enough vs. when you need the full text rewritten to suppress *inferable* identifiers (we'll Replace today and only mention Rewrite at the end).

## The composability story

Across three notebooks: **Data Designer generated** the training data, **the judge filtered** it, **Data Designer generated** the deployment logs (in `bonus_generate_usage_logs.ipynb`), and **Anonymizer protects** the logs so the wider team can study usage without privacy risk. Same ecosystem, same declarative shape, four composable steps.

## Setup

Note the package vs. import naming: dependency is `nemo-anonymizer`, but you import from `anonymizer`. `uv sync` already installed it for you.

In [ ]:
import json
from collections import Counter

import pandas as pd

from notebook_helpers import (
    DATA_DIR,
    ARTIFACT_DIR,
    build_anonymizer_model_setup,
    environment_setup,
    load_usage_logs_seed,
    show_provider_info,
)

# Pick which hosted backend to use. Leave as "auto" to use whichever key is in
# your .env (precedence: NVIDIA -> OpenRouter -> OpenAI), or set explicitly to
# "nvidia", "openrouter", or "openai" to flip between providers without editing
# any other files.
PROVIDER = "auto"

provider = environment_setup(provider=PROVIDER)
show_provider_info(provider)

## Part 1 — Load the deployment logs

We use `data/usage_logs_seed.parquet` — a dataset of synthetic business-document VLM deployment logs generated by [`bonus_generate_usage_logs.ipynb`](bonus_generate_usage_logs.ipynb). Each row is one **multi-turn user session**. The columns:

| Column | Description |
|---|---|
| `user_persona`, `task_intent`, `document_type` | sampler metadata |
| `system_prompt` | application-built system prompt naming the authenticated user, employer, role, project codename — shown for inspection; the canonical log copy lives at `messages[0]` of `conversation_trace` |
| `attached_document_context` | text representation of the rich business document the user uploaded (clinical trial report, financial variance memo, market research brief, product launch readiness plan, operations dashboard export, customer support incident review) — also shown for inspection |
| `conversation_trace` | full multi-turn ChatML trace as a JSON string — **system message at index 0**, then user messages, assistant tool calls, tool results, assistant replies. This is the canonical log shape (matches OpenAI / Nemotron API log format) and the column we'll anonymize. |

### Where does PII enter your logs? Four surfaces — all inside `conversation_trace`.

Real deployment logs leak identifying information through *four* independent surfaces, and they all live inside the trace because that's how production-quality logs are stored:

| Surface | Where in the trace | What an attacker / casual reader sees |
|---|---|---|
| System prompt | `messages[0]` (`role: "system"`) | the user's name, email, employer, project codename — *every single log line* |
| User messages | `messages[*]` with `role: "user"` | task references, occasional colleague names or codenames |
| Tool calls | `messages[*].tool_calls[*].custom.input` | JSON arguments quote organization + document IDs |
| Tool results | `messages[*]` with `role: "tool"` | *additional* PII the user never typed (CRM contacts, owner emails, contract IDs, ARR bands, reviewer names) |

The system prompt and the tool results are the **most under-appreciated** sources — most teams forget the system prompt is in every log, and the tool results contain identifiers the user never sent. Because all four surfaces live inside `conversation_trace`, we anonymize that **single column**. The standalone `system_prompt` and `attached_document_context` columns are kept for inspection but are not separately anonymized; downstream consumers should read from the anonymized `conversation_trace` instead.

In [ ]:
logs = load_usage_logs_seed()
print(f"Loaded {len(logs)} log rows.\n")
logs[["user_persona", "task_intent", "document_type"]].head(5)

In [ ]:
# Show one full row of text content so you can see the leaks before we touch anything.
row = logs.iloc[0]
print("═══ system_prompt (application-built) ═══")
print(row["system_prompt"])
print("\n═══ attached_document_context (uploaded doc) ═══")
print(row["attached_document_context"])
print("\n═══ conversation_trace (multi-turn ChatML, pretty-printed) ═══")
trace = json.loads(row["conversation_trace"])
for i, msg in enumerate(trace["messages"]):
    role = msg.get("role")
    if role == "tool":
        print(f"[{i}] tool         (id={msg.get('tool_call_id')})  {msg.get('content','')[:200]}…")
    elif msg.get("tool_calls"):
        for tc in msg["tool_calls"]:
            cust = tc.get("custom") or {}
            print(f"[{i}] assistant 🛠  {cust.get('name')}({cust.get('input')})")
    else:
        print(f"[{i}] {role:<12} {(msg.get('content') or '')[:200]}")

🔍 **Look at what's in each layer.**

- The **system prompt** alone names the user, their employer, their email, their role, their project codename, and 1–2 colleagues. Every log line carries this string.
- The **attached document** names the organization plus 2–4 stakeholders (often with emails), a document ID, and 3–6 figures (enrollment counts, dollar variances, KPI values, severity levels, etc.).
- The **conversation trace** layers more on top: tool-call argument JSON quotes organization + document IDs; tool results return CRM contacts, owner emails, and reviewer names the user *never typed*; assistant replies echo all of it back.

None of it dramatic on its own — that's the point. Logs leak identifying information through *aggregation* and *casual reference*, not through dramatic disclosures. A human reviewer who reads 5,000 of these is reading the equivalent of someone's analytics-team inbox.

## Part 2 — Configure Anonymizer

Anonymizer is a small, focused tool built on top of Data Designer. Its pipeline:

1. **GLiNER-PII** — a zero-shot NER model scans for entity spans (names, emails, phone numbers, organisations, addresses, amounts, …). Hosted on NVIDIA Build by default, so no local weights. You can also self-host GLiNER if you need fully offline detection.
2. **LLM augmenter / validator** — a text LLM extends coverage on entities GLiNER missed and validates uncertain spans.
3. **Replacement** — applies your chosen strategy to each detected span. `Substitute` calls an LLM per entity; `Redact` / `Annotate` / `Hash` are local string transforms.

### Reusing the same provider plumbing as Data Designer

`anonymizer.ModelProvider` is literally the same class that `data_designer.config.ModelProvider` exports. The Anonymizer constructor takes:

- `model_providers` — the same `list[ModelProvider]` we built for DD
- `model_configs` — a YAML string (or path) defining the model pool plus `selected_models:` blocks naming which alias plays each detector / validator / replacer / rewriter role

`build_anonymizer_model_setup()` in `notebook_helpers.py` renders that YAML for whichever provider the notebook resolved, so the workshop is portable across NVIDIA Build / OpenRouter / OpenAI without rewriting setup.

In [ ]:
from anonymizer import (
    Anonymizer,
    AnonymizerConfig,
    AnonymizerInput,
    Substitute,
)

model_providers, model_configs_yaml = build_anonymizer_model_setup(provider)
print("Anonymizer model_configs YAML:")
print(model_configs_yaml)

anonymizer = Anonymizer(
    model_providers=model_providers,
    model_configs=model_configs_yaml,
)

Anonymizer reads input from a file path or URL, **one text column at a time**. We'll write the loaded dataframe to a temp parquet and create one `AnonymizerInput` per text column.

**Important:** always set `data_summary`. It is the single cheapest quality lever Anonymizer gives you — both detection and replacement use it as context, so the augmenter knows *what kind of text it's looking at*. A one-line description is enough.

In [ ]:
# For the workshop we subsample to 5 rows so each preview/run completes in
# a few minutes against the free tier. Bump LIVE_DEMO_ROWS up (or remove the
# .head() call) when running offline on the full dataset.
LIVE_DEMO_ROWS = 5

tmp_input = (DATA_DIR / "_anonymizer_input.parquet").resolve()
tmp_input.parent.mkdir(parents=True, exist_ok=True)
demo_logs = logs.head(LIVE_DEMO_ROWS).reset_index(drop=True)
demo_logs[["user_persona", "task_intent", "conversation_trace"]].to_parquet(tmp_input, index=False)
print(f"Wrote {len(demo_logs)} rows to {tmp_input.relative_to(DATA_DIR.parent)}.")

DATA_SUMMARY = (
    "Synthetic multi-turn deployment-log traces from a rich-business-document VLM. "
    "Each record is one full ChatML conversation trace stored as a JSON string. "
    "messages[0] is the application's system prompt with authenticated user identity "
    "(name, email, employer, role, project codename, colleagues). The remaining "
    "messages are the multi-turn dialogue: terse user messages referencing the "
    "uploaded document, assistant tool calls (lookup_organization_record, "
    "lookup_owner_contact, compare_to_baseline, validate_compliance, flag_for_review, "
    "notify_stakeholder) with JSON arguments, tool result messages with JSON CRM / "
    "owner / compliance / review responses, and assistant replies grounded in those "
    "results. Free text and embedded JSON contain realistic business identifiers: "
    "organization names, named stakeholders (full name + role + email), document IDs, "
    "dates, dollar variances and KPI figures, addresses, phone numbers, email "
    "addresses, and internal project codenames."
)

TEXT_COLUMN = "conversation_trace"
anonymizer_input = AnonymizerInput(
    source=str(tmp_input),
    text_column=TEXT_COLUMN,
    data_summary=DATA_SUMMARY,
)
print(f"Anonymizer input ready for column: {TEXT_COLUMN!r}")

## Part 3 — Preview with `Substitute`

`Substitute` is the production default for analytics-style log workflows: for each detected entity it asks an LLM to generate a *contextually plausible* replacement. **"Wescott Logistics" might become "Halberg Freight"**. Sentence flow stays natural — the user message still reads like a user message; assistant replies still read like model output. That naturalness preserves analytical signal: you can still read the logs and see *what kind of question the user was asking* and *what the model did with each tool call*.

It costs one LLM call per detected entity, which is why we always start with a `preview()` to inspect detector output before paying for the full run.

`preview()` runs on a small sample. Cheap. Always do this before a full run.

In [ ]:
substitute_config = AnonymizerConfig(replace=Substitute())
substitute_preview = anonymizer.preview(
    config=substitute_config, data=anonymizer_input, num_records=3,
)
substitute_preview.display_record()

🔍 **Inspect what the detector caught.** The Rich-highlighted display shows each detected entity with its label and replacement. Look critically:

- Did organization names and named stakeholders get tagged as `organization` / `person`?
- Were document IDs, monetary variances, and KPI figures recognised?
- Are there **false positives** — something tagged as PII that isn't actually identifying (e.g. a generic chart axis label)?
- Are there **false negatives** — something obviously identifying that slipped through (e.g. an internal project codename, a colleague's first name)?

PII detection is never perfect. The point of `preview()` is that you see this *before* you commit to the full run. If you're seeing systematic gaps, there are knobs: lower `gliner_threshold` for recall (default 0.3, try 0.2), or extend the entity-label list with domain terms via `Detect(entity_labels=[*DEFAULT_ENTITY_LABELS, ...])`.

The next cell makes the value tangible: it runs a Presidio-style regex baseline (the kind of detector most rule-based PII tools ship with out of the box) on the same rows, then reports the gap between what regex caught and what GLiNER + the LLM augment / validate steps surfaced. The gap is the answer to *"why not just regex this away?"*.

In [ ]:
# A "Presidio-style" baseline: the entity types every conventional PII tool
# ships with out-of-the-box. We hand-code the regexes here just to make the
# baseline reproducible without an extra dependency.
import re

PRESIDIO_LIKE_PATTERNS = {
    "EMAIL_ADDRESS":  re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b"),
    "PHONE_NUMBER":   re.compile(r"\+?\d[\d\s().-]{7,}\d"),
    "DATE_TIME":      re.compile(r"\b\d{4}-\d{2}-\d{2}\b"),
    "US_ZIP_CODE":    re.compile(r"\b\d{5}(?:-\d{4})?\b"),
    "US_SSN":         re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
    "IP_ADDRESS":     re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b"),
    "CREDIT_CARD":    re.compile(r"\b(?:\d[ -]?){13,16}\b"),
}

texts = demo_logs[TEXT_COLUMN].astype(str).tolist()
print(f"Presidio-style regex baseline on the same {len(texts)} rows the Substitute preview just ran on:\n")

baseline_total = 0
for label, pat in PRESIDIO_LIKE_PATTERNS.items():
    n = sum(len(pat.findall(t)) for t in texts)
    baseline_total += n
    print(f"  {label:18s}: {n:4d}")
print(f"  {'TOTAL':18s}: {baseline_total:4d}")

# Pull entity counts the Anonymizer pipeline surfaced on the same rows. The
# trace_dataframe carries `final_entities` per row -- a list of dicts after
# detection, LLM-validate, and LLM-augment.
trace_df = substitute_preview.trace_dataframe
if "final_entities" in trace_df.columns:
    all_entities = [e for row in trace_df["final_entities"] if isinstance(row, list) for e in row]
    n_anonymizer = len(all_entities)
    label_breakdown = Counter(ent.get("label", "?") for ent in all_entities)
    print(f"\nAnonymizer (GLiNER + LLM-validate + LLM-augment): {n_anonymizer} entities total")
    print(f"  Top labels: {dict(label_breakdown.most_common(12))}")
    print(f"\n  Gap (entities the regex baseline misses): {n_anonymizer - baseline_total}")
    print(f"  -> Internal project codenames, invented product names, document IDs,")
    print(f"     compound role descriptors, occupations, organizations, etc. --")
    print(f"     categories closed-set regex/dictionary tools simply don't ship with.")
else:
    print(f"\n(Anonymizer entity column not exposed in this version; trace columns: "
          f"{list(trace_df.columns)})")


## Part 4 — A note on `Rewrite` mode (we won't run it here)

Everything above is **Replace mode** -- Anonymizer detects entity spans and replaces them in place. That's the right tool for **structured data where the structure has to survive anonymization**: log traces, JSON tool calls, ChatML messages, OCR'd document fields. After anonymization the JSON still has to parse and the message roles still have to align.

Anonymizer also has a **Rewrite mode** that uses an LLM to rephrase an entire passage against a `PrivacyGoal`. It's the right tool for **free-form text where content matters and structure is flexible** -- narrative customer-support transcripts, free-text reviews, incident-postmortem write-ups, candidate-feedback notes. In those domains a sentence-level rewrite can scrub *inferable* identifiers (writing style, an unusually specific combination of facts) that no span detector can put a box around.

Rewrite mode is **not the right call for this notebook's data**: sentence-level rephrasing would smash the JSON structure of `conversation_trace` into prose. Worth knowing it exists, and worth recognising the use cases where it's the right tool.


## Part 5 — Full run and save

We'll commit to `Substitute` for the final dataset — that's the most common production choice when downstream consumers include models or analytics tooling. Pick differently if your audience is a human audit team.

In [ ]:
# `anonymizer.run()` runs over the full input parquet (no num_records cap).
# On the free-tier NVIDIA Build endpoint a 5-row run on `conversation_trace`
# takes ~3-5 min, so we gate it behind a flag for the live workshop. Flip
# RUN_FULL_PIPELINE to True when running offline or rendering the final artifact.
RUN_FULL_PIPELINE = False

if RUN_FULL_PIPELINE:
    result = anonymizer.run(config=substitute_config, data=anonymizer_input)
    n_failed = len(result.failed_records or [])
    print(f"✓ Anonymised {TEXT_COLUMN}: {len(result.dataframe)} ok, {n_failed} failed")
else:
    print("⏭️  Full run() skipped (RUN_FULL_PIPELINE=False).")
    print("   The preview above shows the same anonymizer behaviour on a sample.")
    print("   To produce the audit-safe parquet for downstream consumers, set RUN_FULL_PIPELINE=True and re-run this cell.")
    # Reuse the substitute preview so the next 'save' cell still has data to write.
    result = substitute_preview

**If you saw any non-zero `failed`** — those are infra issues (rate limits, auth, transient network errors), not a strategy problem. Inspect `results[col].failed_records` to see the per-record reason. Fix that *before* tweaking strategy.

In [ ]:
# Both .run() (full) and .preview() (sample) return the same dataframe shape,
# so the export below works whether you ran the full pipeline above or fell
# back to the substitute preview as a stand-in.
publishable_rows = len(result.dataframe)
publishable = demo_logs.head(publishable_rows).reset_index(drop=True).copy()
publishable[f"{TEXT_COLUMN}_anonymized"] = result.dataframe[f"{TEXT_COLUMN}_replaced"].tolist()

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
# Path is suffix-tagged so the workshop preview run never overwrites the
# canonical full-pipeline artifact (which is produced offline by
# `notebooks/_run_n3_full.py` over all 20 seed rows).
suffix = "full" if RUN_FULL_PIPELINE else "demo_preview"
out_path = ARTIFACT_DIR / f"03_usage_logs_anonymized__{suffix}.parquet"
publishable.to_parquet(out_path, index=False)
print(f"Saved {len(publishable)} anonymised rows to {out_path.relative_to(ARTIFACT_DIR.parent)}")
publishable[["user_persona", "task_intent", TEXT_COLUMN, f"{TEXT_COLUMN}_anonymized"]].head(3)

⚠️  **A note on guarantees.** Anonymizer is best-effort. Detection misses things, especially inferable identifiers and domain-specific terms; replacement is statistical, not certified. For high-stakes datasets you should still run a human spot-check on a sample, set `risk_tolerance="low"` if you move to Rewrite mode, and document the residual risk. *Privacy is a process, not a function call.*

## Recap

Across three notebooks you've built a complete dataset lifecycle:

**sample → seed → generate → judge → filter → anonymise**

All declarative. All version-controlled. The same shape that produced training data in Notebooks 1 and 2 also produced the deployment logs (`bonus_generate_usage_logs.ipynb`), and Anonymizer applies the same declarative idea to privacy: configure, preview, run.

## Where to go next

- **Production-grade VLM SDG:** the [VLM long-document recipes](https://github.com/NVIDIA-NeMo/DataDesigner/tree/main/docs/assets/recipes/vlm_long_doc) on GitHub are the full version of Notebook 2 — eight stages including OCR, classification-first filtering, multi-page windowed QA, whole-document QA, and a frontier judge. Background story is in the [iterative SDG dev note](https://nvidia-nemo.github.io/DataDesigner/latest/devnotes/training-a-vlm-to-understand-long-documents-an-iterative-sdg-story/).
- **More Data Designer recipes:** [text-to-SQL, code generation, agent search trajectories, deep research](https://github.com/NVIDIA-NeMo/DataDesigner/tree/main/docs/assets/recipes).
- **Anonymizer Rewrite mode:** [introducing-nemo-anonymizer](https://nvidia-nemo.github.io/Anonymizer/latest/devnotes/introducing-nemo-anonymizer-text-anonymization-for-the-reasoning-era/) — the dev note covering Rewrite mode, `PrivacyGoal`, and inferable-identifier suppression.

## One principle to take with you

Treat dataset creation — *and the data your model produces in deployment* — as engineering. Distributions, dependencies, validation, filtering, privacy: these are pipeline stages, not afterthoughts. The tools matter less than the discipline. *Your data deserves the same engineering rigour as your model.*

Thanks for coming. 👋